In [17]:
# Run this in Jupyter
import sklearn

requirements_content = f"""fastapi==0.104.1
uvicorn==0.24.0
pydantic==2.4.2
scikit-learn=={sklearn.__version__}
rdkit-pypi==2022.9.5
numpy==1.24.3
joblib==1.2.2
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

print(f"requirements.txt updated with scikit-learn=={sklearn.__version__}")

with open("requirements.txt") as f:
    print(f.read())

requirements.txt updated with scikit-learn==1.2.2
fastapi==0.104.1
uvicorn==0.24.0
pydantic==2.4.2
scikit-learn==1.2.2
rdkit-pypi==2022.9.5
numpy==1.24.3
joblib==1.2.2



In [18]:
import joblib
import numpy as np
import pandas as pd
import sklearn
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print(f"Current scikit-learn version: {sklearn.__version__}")

# Step 1: EGFR dataset
egfr_data = [
    ("Erlotinib",    "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1",          59.0),
    ("Gefitinib",    "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",      33.0),
    ("Lapatinib",    "CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1", 13.0),
    ("Afatinib",     "CN(C)/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OCC1CCN(C)CC1", 0.5),
    ("Osimertinib",  "C=CC(=O)Nc1cc(-n2ccc(C)c2)c(Nc2ncnc3ccc(N(C)CCN(C)C)cc23)cc1", 1.0),
    ("Vandetanib",   "COc1cc2c(=O)[nH]cc(F)c2cc1Nc1nccc(Br)n1",             500.0),
    ("Neratinib",    "C=CC(=O)Nc1ccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3)c2c1", 59.0),
    ("Dacomitinib",  "C=CC(=O)Nc1ccc2ncnc(Nc3ccc(F)c(Cl)c3)c2c1",          6.0),
    ("Canertinib",   "C=CC(=O)Nc1ccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3)c2c1", 7.4),
    ("Pelitinib",    "C=CC(=O)Nc1cc2ncnc(N)c2cc1OCC",                       17.0),
    ("Compound_A1",  "CCOc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCC",             45.0),
    ("Compound_A2",  "c1ccc(Nc2ncnc3ccccc23)cc1",                           120.0),
    ("Compound_A3",  "CCc1nc2ccc(NC(=O)c3ccccc3)cc2s1",                     89.0),
    ("Compound_A4",  "Fc1ccc(Nc2ncnc3ccc(OC)c(OC)c23)cc1Cl",               234.0),
    ("Compound_A5",  "COc1ccc(Nc2ncnc3ccccc23)cc1",                         567.0),
    ("Compound_A6",  "Cc1cc2ncnc(N)c2cc1NC(=O)c1ccccc1",                    89.0),
    ("Compound_A7",  "c1ccc2c(Nc3ccc(Br)cc3)ncnc2c1",                       445.0),
    ("Compound_A8",  "COc1cccc(Nc2ncnc3ccccc23)c1",                         123.0),
    ("Compound_A9",  "CCNc1ncnc2ccc(OC)c(OC)c12",                           678.0),
    ("Compound_A10", "Fc1ccc(CNc2ncnc3ccccc23)cc1",                         234.0),
    ("Inactive_B1",  "CC(=O)Nc1ccc(O)cc1",                                  50000.0),
    ("Inactive_B2",  "CC(C)Cc1ccc(CC(C)C(=O)O)cc1",                         25000.0),
    ("Inactive_B3",  "Cn1c(=O)c2c(ncn2C)n(C)c1=O",                          45000.0),
    ("Inactive_B4",  "OC(=O)c1ccccc1",                                       80000.0),
    ("Inactive_B5",  "CC(=O)Oc1ccccc1C(=O)O",                               35000.0),
    ("Inactive_B6",  "CN(C)C(=N)NC(=N)N",                                   90000.0),
    ("Inactive_B7",  "c1ccc(CC2CCNCC2)cc1",                                  15000.0),
    ("Inactive_B8",  "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",                 20000.0),
    ("Inactive_B9",  "OC1CCCCC1",                                            55000.0),
    ("Inactive_B10", "CCCCCC",                                               100000.0),
    ("Inactive_B11", "c1ccccc1",                                             100000.0),
    ("Inactive_B12", "CCO",                                                  100000.0),
    ("Inactive_B13", "CC(C)O",                                               100000.0),
    ("Inactive_B14", "CCCCO",                                                100000.0),
]

df = pd.DataFrame(egfr_data, columns=["name", "smiles", "ic50_nm"])
df = df[(df["ic50_nm"] <= 1000) | (df["ic50_nm"] >= 10000)].copy()
df["active"] = (df["ic50_nm"] <= 1000).astype(int)
print(f"Active: {df['active'].sum()} | Inactive: {(df['active']==0).sum()}")

# Step 2: Features
def compute_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    desc = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
    ]
    fp = list(AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=2, nBits=1024))
    return desc + fp

features_list = []
valid_idx = []
for i, row in df.iterrows():
    f = compute_features(row["smiles"])
    if f is not None:
        features_list.append(f)
        valid_idx.append(i)

X = np.array(features_list)
y = df["active"].iloc[valid_idx].values
print(f"Feature matrix shape: {X.shape}")

# Step 3: Train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  RandomForestClassifier(
        n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
print(f"Train accuracy: {pipeline.score(X_train, y_train):.2%}")
print(f"Test accuracy:  {pipeline.score(X_test, y_test):.2%}")

# Step 4: Save fresh
joblib.dump(pipeline, "qsar_model.pkl")
print(f"\n✅ Model resaved with scikit-learn=={sklearn.__version__}")

Current scikit-learn version: 1.2.2
Active: 20 | Inactive: 14
Feature matrix shape: (34, 1030)
Train accuracy: 100.00%
Test accuracy:  100.00%

✅ Model resaved with scikit-learn==1.2.2


In [19]:
import os

# Make sure we're in the right folder
print("Current folder:", os.getcwd())

# Create the Dockerfile
dockerfile_content = """FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y \\
    libxrender1 \\
    libxext6 \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .
COPY qsar_model.pkl .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile_content)
print("Dockerfile created!")

# Create requirements.txt (plural, with content)
requirements_content = """fastapi==0.104.1
uvicorn==0.24.0
pydantic==2.4.2
scikit-learn==1.3.2
rdkit-pypi==2022.9.5
numpy==1.24.3
joblib==1.2.2
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)
print("requirements.txt created!")

# Verify everything
print("\nFiles now in folder:")
for f in os.listdir("."):
    size = os.path.getsize(f)
    print(f"  {f} ({size} bytes)")

Current folder: C:\Users\amart\DATA ANALYSIS\Amazon\ml_journey
Dockerfile created!
requirements.txt created!

Files now in folder:
  .ipynb_checkpoints (0 bytes)
  Dockerfile (361 bytes)
  Docker_file.ipynb (13391 bytes)
  main.py (7485 bytes)
  qsar_model.pkl (131183 bytes)
  requirements.txt (125 bytes)


In [20]:
# Run this in Jupyter RIGHT NOW
import sklearn
print(f"Jupyter scikit-learn version: {sklearn.__version__}")

import joblib
model = joblib.load("qsar_model.pkl")
print(f"Model loads fine here: {type(model)}")

# Check numpy version too - this can also cause pickle issues
import numpy as np
print(f"Jupyter numpy version: {np.__version__}")

Jupyter scikit-learn version: 1.2.2
Model loads fine here: <class 'sklearn.pipeline.Pipeline'>
Jupyter numpy version: 1.24.3


In [21]:
# Run this completely in Jupyter
import sklearn
import numpy
import joblib as jl

print(f"scikit-learn: {sklearn.__version__}")
print(f"numpy:        {numpy.__version__}")
print(f"joblib:       {jl.__version__}")

scikit-learn: 1.2.2
numpy:        1.24.3
joblib:       1.2.0


In [22]:
requirements_content = f"""fastapi==0.104.1
uvicorn==0.24.0
pydantic==2.4.2
scikit-learn=={sklearn.__version__}
rdkit-pypi==2022.9.5
numpy=={numpy.__version__}
joblib=={jl.__version__}
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

print("=== requirements.txt content ===")
with open("requirements.txt") as f:
    print(f.read())

=== requirements.txt content ===
fastapi==0.104.1
uvicorn==0.24.0
pydantic==2.4.2
scikit-learn==1.2.2
rdkit-pypi==2022.9.5
numpy==1.24.3
joblib==1.2.0

